In [1]:
# 
# MULTIPANEL FIGURE DATA PREPARATION
#
# This notebook:
# 1. Loads Zarr data for historical and future scenarios
# 2. Calculates departures and percentages
# 3. Runs/loads statistical tests (Mann-Whitney U + FDR)
# 4. Saves all intermediate data as NetCDF for figure generation
#
# Output: NetCDF files in multipanel_percentile/multipanel_{RCP}/


import os
import numpy as np
import xarray as xr
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from distributed import Client
from tqdm.auto import tqdm
import warnings

print("Initializing Dask client...")
client = Client(n_workers=4, threads_per_worker=4, memory_limit="32GB")
print(f"Dask client initialized: {client}")

Initializing Dask client...


/anaconda3/envs/pyEAE/lib/python3.9/site-packages/distributed/node.py:179: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 40781 instead
  warnings.warn(


Dask client initialized: <Client: 'tcp://127.0.0.1:39279' processes=4 threads=16, memory=119.21 GiB>


In [17]:
# PATH CONFIGURATION - EDIT THIS SECTION
# Set DATA_ROOT to the location where you extracted SDHI_Precip_supporting_data.tar.gz
DATA_ROOT = "/path/to/SDHI_Precip_supporting_data"  # <-- EDIT THIS LINE, if notebook is in root you can use "./"

# Derived paths (no editing needed below this line)
BASE_PATH_ANNUAL = f"{DATA_ROOT}/combined_datasets"
BASE_PATH_SEASONAL = f"{DATA_ROOT}/combined_seasonal_datasets"
PATH_GRID = f"{DATA_ROOT}/data"

# Statistical test cache (intermediate - stored in DATA_ROOT for reuse)
# These are expensive to compute, so we cache them
STATS_CACHE_BASE = f"{DATA_ROOT}/stats_cache"

# ANALYSIS CONFIGURATION
RCP_scenario = '8p5'  # options: "4p5" or "8p5"
percentile_band = '99-100'  # Matches Zarr label format

# Intermediate outputs (used by notebook 32)
# Stored relative to notebook location
output_base = './multipanel_percentile'
output_dir = f"{output_base}/multipanel_{RCP_scenario}"

# Create output subdirectories
for subdir in ['departures', 'percentages', 'statistics', 'hist_means', 'futr_means']:
    os.makedirs(f"{output_dir}/{subdir}", exist_ok=True)

# Statistical test cache location (in DATA_ROOT for reuse)
stats_cache_dir = f'{STATS_CACHE_BASE}/EOC{RCP_scenario}/epoch_aggregated/statistical_analysis'
os.makedirs(stats_cache_dir, exist_ok=True)

recalculate_stats = False  # Set True to force recalculation

print(f"\nConfiguration:")
print(f"  RCP Scenario: {RCP_scenario}")
print(f"  Percentile band: {percentile_band}")
print(f"  Output directory: {output_dir}")
print(f"  Stats cache directory: {stats_cache_dir}")
print(f"  Recalculate stats: {recalculate_stats}")


Configuration:
  RCP Scenario: 8p5
  Percentile band: 99-100
  Output directory: ./multipanel_percentile/multipanel_8p5
  Stats cache directory: .//stats_cache/EOC8p5/epoch_aggregated/statistical_analysis
  Recalculate stats: False


In [18]:
# FILE PATHS

hist_paths = {
    'annual': f'{BASE_PATH_ANNUAL}/dist_hist_dataset.zarr',
    'mam': f'{BASE_PATH_SEASONAL}/dist_hist_MAM_dataset.zarr',
    'jja': f'{BASE_PATH_SEASONAL}/dist_hist_JJA_dataset.zarr',
    'son': f'{BASE_PATH_SEASONAL}/dist_hist_SON_dataset.zarr',
    'djf': f'{BASE_PATH_SEASONAL}/dist_hist_DJF_dataset.zarr'
}

futr_paths = {
    'annual': f'{BASE_PATH_ANNUAL}/dist_eoc{RCP_scenario}_dataset.zarr',
    'mam': f'{BASE_PATH_SEASONAL}/dist_eoc{RCP_scenario}_MAM_dataset.zarr',
    'jja': f'{BASE_PATH_SEASONAL}/dist_eoc{RCP_scenario}_JJA_dataset.zarr',
    'son': f'{BASE_PATH_SEASONAL}/dist_eoc{RCP_scenario}_SON_dataset.zarr',
    'djf': f'{BASE_PATH_SEASONAL}/dist_eoc{RCP_scenario}_DJF_dataset.zarr'
}


In [19]:
# HELPER FUNCTIONS

def normalize_dimension_names(ds):
    """Normalize dimension names (concat_dim -> Time)"""
    if 'concat_dim' in ds.dims:
        ds = ds.rename({'concat_dim': 'Time'})
        print(f"      Renamed 'concat_dim' to 'Time'")
    return ds


def load_datasets():
    """Load historical and future datasets"""
    print("\nLoading datasets...")
    hist_datasets = {}
    futr_datasets = {}
    
    # Load historical
    print("Loading historical datasets:")
    for season, path in tqdm(hist_paths.items(), desc="Historical"):
        print(f"\n  {season.upper()}: {os.path.basename(path)}")
        
        try:
            ds = xr.open_dataset(path, chunks='auto', engine='zarr')
        except ValueError:
            import zarr
            ds = xr.open_zarr(zarr.DirectoryStore(path))
        
        print(f"    Loaded shape: {dict(ds.sizes)}")
        
        # Filter to percentile band
        if 'percentile_label' in ds.coords:
            print(f"    Filtering to percentile band: {percentile_band}")
            mask = (ds['percentile_label'] == percentile_band).compute()
            valid_indices = mask.values.nonzero()[0]
            print(f"    Found {len(valid_indices)} matching records")
            
            if len(valid_indices) == 0:
                print(f"    WARNING: No data for percentile band {percentile_band}!")
                print(f"    Available percentile labels: {np.unique(ds['percentile_label'].values)}")
            
            ds = ds.isel(concat_dim=valid_indices)
            print(f"    After filtering shape: {dict(ds.sizes)}")
        elif 'percentile_band' in ds.coords:
            print(f"    Using percentile_band coordinate")
            ds = ds.sel(percentile_band=percentile_band)
        else:
            print(f"    No percentile filtering coordinate found")
        
        ds = normalize_dimension_names(ds)
        print(f"    Final shape: {dict(ds.sizes)}")
        
        hist_datasets[season] = ds
    
    # Load future
    print("\nLoading future datasets:")
    for season, path in tqdm(futr_paths.items(), desc="Future"):
        print(f"\n  {season.upper()}: {os.path.basename(path)}")
        
        try:
            ds = xr.open_dataset(path, chunks='auto', engine='zarr')
        except ValueError:
            import zarr
            ds = xr.open_zarr(zarr.DirectoryStore(path))
        
        print(f"    Loaded shape: {dict(ds.sizes)}")
        
        # Filter to percentile band
        if 'percentile_label' in ds.coords:
            mask = (ds['percentile_label'] == percentile_band).compute()
            valid_indices = mask.values.nonzero()[0]
            print(f"    Found {len(valid_indices)} matching records")
            
            if len(valid_indices) == 0:
                print(f"    WARNING: No data for percentile band {percentile_band}!")
            
            ds = ds.isel(concat_dim=valid_indices)
        elif 'percentile_band' in ds.coords:
            ds = ds.sel(percentile_band=percentile_band)
        
        ds = normalize_dimension_names(ds)
        print(f"    Final shape: {dict(ds.sizes)}")
        
        futr_datasets[season] = ds
    
    # Add coordinates
    print("\nAdding lat/lon coordinates...")
    grid = xr.open_dataset(f"{PATH_GRID}/geo_em.d01.nc", engine='netcdf4')
    lat_coords = grid.XLAT_M[0, :, :].values
    lon_coords = grid.XLONG_M[0, :, :].values
    
    print(f"Grid coordinates shape: {lat_coords.shape}")
    
    for ds_dict in [hist_datasets, futr_datasets]:
        for season, ds in ds_dict.items():
            ds.coords['lat'] = (['south_north', 'west_east'], lat_coords)
            ds.coords['lon'] = (['south_north', 'west_east'], lon_coords)
    
    print("Dataset loading complete.")
    return hist_datasets, futr_datasets


def apply_mannwhitneyu(data0, data1):
    """Apply Mann-Whitney U test gridcell by gridcell"""
    print(f"    Running Mann-Whitney U test...")
    
    flat_data0 = data0.values.reshape((data0.shape[0], -1))
    flat_data1 = data1.values.reshape((data1.shape[0], -1))
    
    p_value = np.empty_like(flat_data0[0], dtype=float)
    
    for i in tqdm(range(len(p_value)), desc="    Grid cells", leave=False):
        try:
            _, p_value[i] = mannwhitneyu(flat_data0[:, i], flat_data1[:, i], alternative='two-sided')
        except:
            p_value[i] = np.nan
    
    p_value_2d = p_value.reshape(data0.shape[1:])
    return xr.Dataset({'p_value': (('south_north', 'west_east'), p_value_2d)})


def fdr_adjust(ds):
    """Apply FDR correction"""
    print(f"    Applying FDR correction...")
    p_values = ds.p_value.values.flatten()
    valid_mask = ~np.isnan(p_values)
    valid_p_values = p_values[valid_mask]
    
    if len(valid_p_values) > 0:
        _, corrected_valid, _, _ = multipletests(valid_p_values, method='fdr_bh')
        corrected_p_values = p_values.copy()
        corrected_p_values[valid_mask] = corrected_valid
    else:
        corrected_p_values = p_values
    
    corrected_p_values = corrected_p_values.reshape(ds.p_value.shape)
    
    ds_fdr = xr.Dataset({
        'p_value_fdr_corrected': (('south_north', 'west_east'), corrected_p_values)
    })
    ds_fdr.coords['lat'] = ds.lat
    ds_fdr.coords['lon'] = ds.lon
    
    return ds_fdr


In [21]:
# MAIN PROCESSING

print("STEP 1: LOAD DATA")

hist_ds, futr_ds = load_datasets()



# DIAGNOSTIC: Check loaded data

print("DATA DIAGNOSTICS")

for season in ['annual']:  # Just check annual first
    print(f"\n{season.upper()} diagnostics:")
    
    hist = hist_ds[season]
    print(f"  Historical dataset structure:")
    print(f"    Variables: {list(hist.data_vars)}")
    print(f"    Dimensions: {dict(hist.dims)}")
    
    # Check what happened with Time dimension
    if 'Time' in hist.dims:
        print(f"\n  Time dimension size: {hist.dims['Time']}")
        if hist.dims['Time'] == 0:
            print(f"  ERROR: Time dimension is ZERO!")
            print(f"  (This likely means the percentile filtering removed all data)")
            
            # Load the raw data again to check available labels
            print(f"\n  Checking available percentile labels in raw data...")
            raw_path = hist_paths['annual']
            try:
                raw_ds = xr.open_dataset(raw_path, chunks='auto', engine='zarr')
            except ValueError:
                import zarr
                raw_ds = xr.open_zarr(zarr.DirectoryStore(raw_path))
            
            if 'percentile_label' in raw_ds.coords:
                print(f"\n  Available percentile labels:")
                unique_labels = np.unique(raw_ds['percentile_label'].values)
                for label in unique_labels:
                    count = (raw_ds['percentile_label'] == label).sum().values
                    print(f"    '{label}': {count} records")
                
                print(f"\n  Searching for: '{percentile_band}'")
                print(f"  Does the percentile interval (band) exist? {percentile_band in unique_labels}")
            
            print("\n" + "="*70)
            print("STOPPING - NEED TO FIX PERCENTILE LABEL")
            print("="*70)
            raise ValueError("No data after percentile filtering - check percentile_band variable")
        else:
            # Time dimension has data - continue with checks
            print(f"  Good! Time dimension has {hist.dims['Time']} records")
            
            if 'PRECIP_PERIOD' in hist.data_vars:
                data = hist.PRECIP_PERIOD
                print(f"\n  PRECIP_PERIOD data check:")
                print(f"    Shape: {data.shape}")
                
                # Compute a small sample to check
                sample = data.isel(Time=0, south_north=slice(400,410), west_east=slice(600,610)).compute()
                print(f"    Sample (Time=0, 10x10 region):")
                print(f"      Min: {sample.min().values}, Max: {sample.max().values}, Mean: {sample.mean().values}")
                
                # Check full stats
                print(f"\n  Computing full stats...")
                print(f"    Min: {data.min().compute().values}")
                print(f"    Max: {data.max().compute().values}")
                print(f"    Mean: {data.mean().compute().values}")

print("STEP 2: CALCULATE DEPARTURES AND PERCENTAGES")

seasons = ['annual', 'mam', 'jja', 'son', 'djf']

for season in tqdm(seasons, desc="Processing seasons"):
    print(f"\nProcessing {season.upper()}...")
    
    # Calculate means
    print("  Computing temporal means...")
    hist_mean = hist_ds[season].mean(dim="Time")
    futr_mean = futr_ds[season].mean(dim="Time")
    
    # COMPUTE before checking
    print("  Computing dask arrays...")
    hist_mean = hist_mean.compute()
    futr_mean = futr_mean.compute()
    
    # Quick check
    print(f"    Hist mean range: [{hist_mean.PRECIP_PERIOD.min().values:.2f}, {hist_mean.PRECIP_PERIOD.max().values:.2f}]")
    print(f"    Futr mean range: [{futr_mean.PRECIP_PERIOD.min().values:.2f}, {futr_mean.PRECIP_PERIOD.max().values:.2f}]")
    
    # Calculate departures
    print("  Computing departures...")
    departure = futr_mean - hist_mean
    
    # Calculate percentages
    print("  Computing percentages...")
    percentage = ((departure) / hist_mean) * 100
    
    # Add metadata
    hist_mean.attrs.update({
        'description': f'{season.upper()} historical mean precipitation periods',
        'scenario': 'HIST',
        'percentile_band': percentile_band,
        'units': 'count'
    })
    
    futr_mean.attrs.update({
        'description': f'{season.upper()} future mean precipitation periods',
        'scenario': f'EOC{RCP_scenario}',
        'percentile_band': percentile_band,
        'units': 'count'
    })
    
    departure.attrs.update({
        'description': f'{season.upper()} departure (future - historical)',
        'comparison': f'EOC{RCP_scenario} - HIST',
        'percentile_band': percentile_band,
        'units': 'count'
    })
    
    percentage.attrs.update({
        'description': f'{season.upper()} percentage change',
        'comparison': f'EOC{RCP_scenario} - HIST',
        'percentile_band': percentile_band,
        'units': 'percent'
    })
    
    # Save to NetCDF with overwrite mode
    print("  Saving to NetCDF...")
    hist_mean.to_netcdf(f"{output_dir}/hist_means/{season}_hist_mean.nc", mode='w')
    futr_mean.to_netcdf(f"{output_dir}/futr_means/{season}_futr_mean.nc", mode='w')
    departure.to_netcdf(f"{output_dir}/departures/{season}_departure.nc", mode='w')
    percentage.to_netcdf(f"{output_dir}/percentages/{season}_percentage.nc", mode='w')
    
    print(f"  {season.upper()} saved")

print("STEP 3: STATISTICAL TESTS")

for season in tqdm(seasons, desc="Statistical tests"):
    print(f"\nProcessing {season.upper()}...")
    
    stats_file = f"{output_dir}/statistics/{season}_mwu_fdr.nc"
    
    # Check if cached stats exist (note: cache uses old '99th-100th' format in filename)
    # We need to search for files that might match
    cached_mwu = None
    cached_fdr = None
    
    # Try both old and new naming conventions
    for band_name in [percentile_band, '99th-100th', '99th percentile']:
        test_mwu = f"{stats_cache_dir}/MWU_HIST-EOC{RCP_scenario}_{band_name}_WetPeriod_0p254base_{season}.nc"
        test_fdr = f"{stats_cache_dir}/MWU_FDR_HIST-EOC{RCP_scenario}_{band_name}_WetPeriod_0p254base_{season}.nc"
        if os.path.exists(test_mwu) and os.path.exists(test_fdr):
            cached_mwu = test_mwu
            cached_fdr = test_fdr
            break
    
    if not recalculate_stats and cached_mwu and cached_fdr:
        print(f"  Loading cached statistics...")
        mwu_ds = xr.open_dataset(cached_mwu)
        fdr_ds = xr.open_dataset(cached_fdr)
    else:
        print(f"  Computing new statistics...")
        mwu_ds = apply_mannwhitneyu(hist_ds[season].PRECIP_PERIOD, futr_ds[season].PRECIP_PERIOD)
        mwu_ds.coords['lat'] = hist_ds[season].coords['lat']
        mwu_ds.coords['lon'] = hist_ds[season].coords['lon']
        
        fdr_ds = fdr_adjust(mwu_ds)
    
    # Combine MWU and FDR into single file
    combined_stats = xr.Dataset({
        'p_value': mwu_ds.p_value,
        'p_value_fdr_corrected': fdr_ds.p_value_fdr_corrected
    })
    combined_stats.coords['lat'] = mwu_ds.lat
    combined_stats.coords['lon'] = mwu_ds.lon
    
    combined_stats.attrs.update({
        'description': f'{season.upper()} statistical test results',
        'comparison': f'HIST vs EOC{RCP_scenario}',
        'percentile_band': percentile_band,
        'test': 'Mann-Whitney U (two-sided)',
        'correction': 'FDR (Benjamini-Hochberg)'
    })
    
    # Save with overwrite mode
    print(f"  Saving statistics...")
    combined_stats.to_netcdf(stats_file, mode='w')
    print(f"  {season.upper()} statistics saved")

print("DATA PREPARATION COMPLETE")
print(f"\nAll files saved to: {output_dir}/")
print("\nOutput structure:")
print(f"  departures/     - {len(seasons)} files")
print(f"  percentages/    - {len(seasons)} files")
print(f"  statistics/     - {len(seasons)} files")
print(f"  hist_means/     - {len(seasons)} files")
print(f"  futr_means/     - {len(seasons)} files")
print(f"\nTotal: {len(seasons) * 5} NetCDF files ready for figure generation")


STEP 1: LOAD DATA

Loading datasets...
Loading historical datasets:


Historical:   0%|          | 0/5 [00:00<?, ?it/s]


  ANNUAL: dist_hist_dataset.zarr
    Loaded shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Filtering to percentile band: 99-100
    Found 15 matching records
    After filtering shape: {'concat_dim': 15, 'south_north': 899, 'west_east': 1399}
      Renamed 'concat_dim' to 'Time'
    Final shape: {'Time': 15, 'south_north': 899, 'west_east': 1399}

  MAM: dist_hist_MAM_dataset.zarr
    Loaded shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Filtering to percentile band: 99-100
    Found 15 matching records
    After filtering shape: {'concat_dim': 15, 'south_north': 899, 'west_east': 1399}
      Renamed 'concat_dim' to 'Time'
    Final shape: {'Time': 15, 'south_north': 899, 'west_east': 1399}

  JJA: dist_hist_JJA_dataset.zarr
    Loaded shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Filtering to percentile band: 99-100
    Found 15 matching records
    After filtering shape: {'concat_dim': 15, 'south_north': 899, 'wes

Future:   0%|          | 0/5 [00:00<?, ?it/s]


  ANNUAL: dist_eoc8p5_dataset.zarr
    Loaded shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Found 15 matching records
      Renamed 'concat_dim' to 'Time'
    Final shape: {'Time': 15, 'south_north': 899, 'west_east': 1399}

  MAM: dist_eoc8p5_MAM_dataset.zarr
    Loaded shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Found 15 matching records
      Renamed 'concat_dim' to 'Time'
    Final shape: {'Time': 15, 'south_north': 899, 'west_east': 1399}

  JJA: dist_eoc8p5_JJA_dataset.zarr
    Loaded shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Found 15 matching records
      Renamed 'concat_dim' to 'Time'
    Final shape: {'Time': 15, 'south_north': 899, 'west_east': 1399}

  SON: dist_eoc8p5_SON_dataset.zarr
    Loaded shape: {'concat_dim': 180, 'south_north': 899, 'west_east': 1399}
    Found 15 matching records
      Renamed 'concat_dim' to 'Time'
    Final shape: {'Time': 15, 'south_north': 899, 'west_east': 1399}



...

    Applying FDR correction...
  Saving statistics...
  JJA statistics saved

Processing SON...
  Computing new statistics...
    Running Mann-Whitney U test...


    Grid cells:   0%|          | 0/1257701 [00:00<?, ?it/s]

    Applying FDR correction...
  Saving statistics...
  SON statistics saved

Processing DJF...
  Computing new statistics...
    Running Mann-Whitney U test...


    Grid cells:   0%|          | 0/1257701 [00:00<?, ?it/s]

    Applying FDR correction...
  Saving statistics...
  DJF statistics saved
DATA PREPARATION COMPLETE

All files saved to: ./multipanel_percentile/multipanel_8p5/

Output structure:
  departures/     - 5 files
  percentages/    - 5 files
  statistics/     - 5 files
  hist_means/     - 5 files
  futr_means/     - 5 files

Total: 25 NetCDF files ready for figure generation
